In [ ]:
!git clone -b week4/deepfake https://github.com/eth-bmai-fs26/project.git

In [ ]:
# Deepfake Discriminator — Real vs Fake Detection
# Two approaches: (1) Supervised CNN, (2) Reconstruction-based anomaly detection

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)
import os, shutil, random, zipfile

import sys
# Ensure utils.py is importable regardless of working directory (e.g. Colab)
_nb_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
# If running from repo root, point to the notebook's directory
for _candidate in [_nb_dir, os.path.join(_nb_dir, 'week4', 'deepfake')]:
    if os.path.exists(os.path.join(_candidate, 'utils.py')):
        sys.path.insert(0, _candidate)
        break

from utils import (
    SEED, IMG_SIZE, prepare_fruit_dataset, load_fruit_data,
    collect_all_images,
)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Deepfake Detection: Supervised vs Representation Learning

## Goal
Detect whether a fruit image is **real** or **generated by diffusion**.

## Two Approaches

### Approach 1 — Supervised CNN Discriminator
A binary classifier trained on real + fake images with a **~20:1 class imbalance**, simulating the real-world constraint where confirmed fakes are scarce.

### Approach 2 — Reconstruction-Based Anomaly Detection (MAE-style)
A convolutional autoencoder trained on **real images only**. It learns the manifold of real fruit images. At test time, fakes reconstruct poorly because they lie off-manifold — the reconstruction error becomes the anomaly score. This approach needs **zero fake training samples**.

This connects to the course themes of **representation learning**, **manifolds/VAEs**, and **masked autoencoders**.

In [ ]:
# Constants
BATCH_SIZE = 32

# Discriminator-specific
DISC_BATCH_SIZE = 16
DISC_EPOCHS = 30
DISC_LR = 1e-4
N_FAKES = 100
N_FAKE_TRAIN = 60
N_FAKE_VAL = 15
N_FAKE_TEST = 25

# Autoencoder-specific
AE_EPOCHS = 40
AE_LR = 1e-3

In [ ]:
# === Upload and unzip deepfake images ===
# Generated by the last cell of attention_diffusion_pipeline.ipynb

FAKE_DIR = 'deepfake_imgs'

if not os.path.exists(FAKE_DIR):
    try:
        from google.colab import files
        print("Upload deepfake_imgs.zip:")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
    except ImportError:
        zip_name = 'deepfake_imgs.zip'
        print(f"Not in Colab — looking for {zip_name} locally")

    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('.')
    print(f"Extracted to {FAKE_DIR}/")
else:
    print(f"{FAKE_DIR}/ already exists, skipping upload")

# Load fake images as tensors
fake_files = sorted([f for f in os.listdir(FAKE_DIR) if f.endswith('.png')])
print(f"Found {len(fake_files)} fake images")

to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

fake_images = []
for fname in fake_files:
    img = Image.open(os.path.join(FAKE_DIR, fname)).convert('RGB')
    fake_images.append(to_tensor(img))
fake_images = torch.stack(fake_images)

print(f"Fake images tensor: {fake_images.shape}")

In [ ]:
# === Load Fruits-360 dataset (real images) ===

!pip install -q kagglehub
import kagglehub

dataset_path = kagglehub.dataset_download("moltean/fruits")
print(f"Dataset downloaded to: {dataset_path}")

train_dataset, test_dataset, train_loader, test_loader, CLASS_NAMES = \
    load_fruit_data(dataset_path, batch_size=BATCH_SIZE)

In [ ]:
# === Visualize real vs fake samples ===

real_sample, _ = next(iter(train_loader))
n_show = 8

fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))
for i in range(n_show):
    axes[0, i].imshow(real_sample[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(fake_images[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Real', fontsize=13, rotation=0, labelpad=40, va='center')
axes[1, 0].set_ylabel('Fake', fontsize=13, rotation=0, labelpad=40, va='center')
plt.suptitle('Real vs Generated (Fake) Samples', fontsize=14)
plt.tight_layout()
plt.show()

---
## Approach 1: Supervised CNN Discriminator
Binary classifier trained with weighted loss to handle the 20:1 real-to-fake imbalance.

In [ ]:
# === Discriminator dataset and model ===

class RealFakeDataset(Dataset):
    """Binary dataset: real images (label=0) and fake images (label=1)."""

    def __init__(self, real_images, fake_images, augment_fakes=False):
        self.images = torch.cat([real_images, fake_images], dim=0)
        self.labels = torch.cat([
            torch.zeros(len(real_images)),
            torch.ones(len(fake_images)),
        ])
        self.augment_fakes = augment_fakes
        self.n_real = len(real_images)
        self.fake_aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.augment_fakes and idx >= self.n_real:
            img = self.fake_aug(img)
        return img, label


class DeepfakeDiscriminator(nn.Module):
    """
    Lightweight CNN binary classifier: real (0) vs fake (1).
    Input: (B, 3, 64, 64) -> Output: (B, 1) logit.
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.GroupNorm(8, 32),
            nn.LeakyReLU(0.2),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.GroupNorm(8, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.25),

            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.GroupNorm(8, 256),
            nn.LeakyReLU(0.2),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


disc = DeepfakeDiscriminator().to(device)
print(f"Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}")

In [ ]:
# === Create train / val / test splits ===

real_train_all = collect_all_images(train_loader)
real_test_all = collect_all_images(test_loader)

# Split real training images into train and val (80/20)
n_real_total = len(real_train_all)
n_real_val = int(n_real_total * 0.2)

indices = torch.randperm(n_real_total)
real_train = real_train_all[indices[:n_real_total - n_real_val]]
real_val = real_train_all[indices[n_real_total - n_real_val:]]

# Split fake images
fake_train = fake_images[:N_FAKE_TRAIN]
fake_val = fake_images[N_FAKE_TRAIN:N_FAKE_TRAIN + N_FAKE_VAL]
fake_test = fake_images[N_FAKE_TRAIN + N_FAKE_VAL:]

print(f"Train: {len(real_train)} real + {len(fake_train)} fake  (ratio {len(real_train)/len(fake_train):.0f}:1)")
print(f"Val:   {len(real_val)} real + {len(fake_val)} fake  (ratio {len(real_val)/len(fake_val):.0f}:1)")
print(f"Test:  {len(real_test_all)} real + {len(fake_test)} fake  (ratio {len(real_test_all)/len(fake_test):.0f}:1)")

train_ds = RealFakeDataset(real_train, fake_train, augment_fakes=True)
val_ds = RealFakeDataset(real_val, fake_val, augment_fakes=False)
test_ds = RealFakeDataset(real_test_all, fake_test, augment_fakes=False)

disc_train_loader = DataLoader(train_ds, batch_size=DISC_BATCH_SIZE, shuffle=True)
disc_val_loader = DataLoader(val_ds, batch_size=DISC_BATCH_SIZE, shuffle=False)
disc_test_loader = DataLoader(test_ds, batch_size=DISC_BATCH_SIZE, shuffle=False)

In [ ]:
# === CNN Discriminator training ===

pos_weight = torch.tensor([len(real_train) / len(fake_train)]).to(device)
print(f"pos_weight = {pos_weight.item():.1f} (fake samples weighted {pos_weight.item():.0f}x)")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(disc.parameters(), lr=DISC_LR, weight_decay=1e-4)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

disc_train_losses = []
disc_val_losses = []
best_val_loss = float('inf')
best_state = None

for epoch in range(DISC_EPOCHS):
    disc.train()
    epoch_loss = 0.0
    n_batches = 0
    for images, labels in disc_train_loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        logits = disc(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_train = epoch_loss / n_batches
    disc_train_losses.append(avg_train)

    disc.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for images, labels in disc_val_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            val_loss += criterion(disc(images), labels).item()
            n_val += 1

    avg_val = val_loss / n_val
    disc_val_losses.append(avg_val)
    lr_scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.cpu().clone() for k, v in disc.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{DISC_EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}")

disc.load_state_dict(best_state)
disc = disc.to(device)
print(f"\nRestored best model (val_loss={best_val_loss:.4f})")

In [ ]:
# === CNN Discriminator loss curves ===

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(disc_train_losses, label='Train Loss', linewidth=2)
ax.plot(disc_val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss (weighted)')
ax.set_title('Approach 1: CNN Discriminator Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# === CNN Discriminator evaluation ===

@torch.no_grad()
def evaluate_binary(labels, probs, title=""):
    """Compute and plot metrics for a binary classifier."""
    preds = (probs > 0.5).astype(int)

    print("=" * 50)
    print(f"CLASSIFICATION REPORT — {title}")
    print("=" * 50)
    print(classification_report(labels, preds,
          target_names=['Real', 'Fake'], digits=3))

    auc_score = roc_auc_score(labels, probs)
    ap = average_precision_score(labels, probs)
    print(f"ROC-AUC: {auc_score:.4f}")
    print(f"Average Precision (PR-AUC): {ap:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0].set_title('Confusion Matrix', fontsize=13)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Real', 'Fake'])
    axes[0].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]),
                        ha='center', va='center', fontsize=16,
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
    fig.colorbar(im, ax=axes[0], fraction=0.046)

    # ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_score:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontsize=13)
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    axes[2].plot(recall, precision, linewidth=2, label=f'AP = {ap:.3f}')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve', fontsize=13)
    axes[2].legend(fontsize=12)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'{title} — Test Set Performance', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

    return probs, preds, auc_score, ap


# Run CNN evaluation
disc.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for images, labels in disc_test_loader:
        logits = disc(images.to(device)).squeeze(1)
        all_logits.append(logits.cpu())
        all_labels.append(labels)

cnn_labels = torch.cat(all_labels).numpy().astype(int)
cnn_probs = torch.sigmoid(torch.cat(all_logits)).numpy()

cnn_probs, cnn_preds, cnn_auc, cnn_ap = evaluate_binary(
    cnn_labels, cnn_probs, "Approach 1: CNN Discriminator"
)

In [ ]:
# === CNN qualitative gallery ===

def show_gallery(test_ds, probs, preds, labels, title="", n_per_row=5):
    fake_mask = labels == 1
    real_mask = labels == 0

    tp_idx = np.where((preds == 1) & fake_mask)[0]
    fn_idx = np.where((preds == 0) & fake_mask)[0]
    fp_idx = np.where((preds == 1) & real_mask)[0]

    categories = [
        ("Correctly Detected Fakes (TP)", tp_idx, 'green'),
        ("Missed Fakes (FN)", fn_idx, 'red'),
        ("False Alarms (FP)", fp_idx, 'orange'),
    ]

    fig, axes = plt.subplots(3, n_per_row, figsize=(3 * n_per_row, 9))

    for row, (cat_title, indices, color) in enumerate(categories):
        n_show = min(n_per_row, len(indices))
        if len(indices) > 0:
            sorted_idx = indices[np.argsort(-probs[indices])] if row == 0 else indices[np.argsort(probs[indices])]
        else:
            sorted_idx = indices

        for col in range(n_per_row):
            ax = axes[row, col]
            if col < n_show:
                idx = sorted_idx[col]
                img = test_ds.images[idx]
                ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
                ax.set_title(f"p={probs[idx]:.2f}", fontsize=9, color=color)
            ax.axis('off')

        axes[row, 0].set_ylabel(f"{cat_title}\n(n={len(indices)})",
                                fontsize=9, rotation=0, labelpad=120, va='center',
                                color=color, fontweight='bold')

    plt.suptitle(f'{title} — Prediction Gallery', fontsize=14)
    plt.tight_layout()
    plt.show()

show_gallery(test_ds, cnn_probs, cnn_preds, cnn_labels, "CNN Discriminator")

---
## Approach 2: Reconstruction-Based Anomaly Detection

**Key insight from representation learning**: A model that learns to reconstruct real images captures the *manifold* of real data. Generated images lie off this manifold, so they reconstruct poorly.

- Train a convolutional autoencoder on **real images only** (no fakes needed!)
- At test time, measure **reconstruction error** (MSE per image)
- Real images → low error (on-manifold), Fakes → high error (off-manifold)
- This connects directly to **VAEs**, **manifolds**, and **MAE** from the course

In [ ]:
# === Convolutional Autoencoder ===

class ConvAutoencoder(nn.Module):
    """
    Convolutional autoencoder that learns to reconstruct real fruit images.
    The bottleneck forces a compressed representation — the learned manifold.
    Images that don't lie on this manifold (fakes) will reconstruct poorly.
    """
    def __init__(self, latent_dim=64):
        super().__init__()

        # Encoder: 64x64x3 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),    # -> 32x32x32
            nn.GroupNorm(8, 32),
            nn.GELU(),

            nn.Conv2d(32, 64, 4, stride=2, padding=1),   # -> 16x16x64
            nn.GroupNorm(8, 64),
            nn.GELU(),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # -> 8x8x128
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.Conv2d(128, 256, 4, stride=2, padding=1), # -> 4x4x256
            nn.GroupNorm(8, 256),
            nn.GELU(),

            nn.Flatten(),                                  # -> 4096
            nn.Linear(256 * 4 * 4, latent_dim),           # -> latent_dim
        )

        # Decoder: latent_dim -> 64x64x3
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # -> 8x8
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # -> 16x16
            nn.GroupNorm(8, 64),
            nn.GELU(),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # -> 32x32
            nn.GroupNorm(8, 32),
            nn.GELU(),

            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # -> 64x64
            nn.Sigmoid(),  # output in [0, 1] to match image range
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        x = self.decoder_fc(z)
        x = x.view(-1, 256, 4, 4)
        return self.decoder(x)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)


autoencoder = ConvAutoencoder().to(device)
print(f"Autoencoder parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")

In [ ]:
# === Train autoencoder on REAL images only ===
# This is the key: the model learns what real images look like.
# It never sees a single fake during training.

ae_optimizer = optim.Adam(autoencoder.parameters(), lr=AE_LR, weight_decay=1e-5)
ae_scheduler = optim.lr_scheduler.CosineAnnealingLR(ae_optimizer, T_max=AE_EPOCHS)

ae_train_losses = []
ae_val_losses = []

# Use the real-only train/val splits from the CNN section
real_train_loader = DataLoader(
    torch.utils.data.TensorDataset(real_train),
    batch_size=BATCH_SIZE, shuffle=True
)
real_val_loader = DataLoader(
    torch.utils.data.TensorDataset(real_val),
    batch_size=BATCH_SIZE, shuffle=False
)

for epoch in range(AE_EPOCHS):
    autoencoder.train()
    epoch_loss = 0.0
    n_batches = 0
    for (images,) in real_train_loader:
        images = images.to(device)
        recon = autoencoder(images)
        loss = F.mse_loss(recon, images)

        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    ae_scheduler.step()
    avg_train = epoch_loss / n_batches
    ae_train_losses.append(avg_train)

    # Validation
    autoencoder.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for (images,) in real_val_loader:
            images = images.to(device)
            val_loss += F.mse_loss(autoencoder(images), images).item()
            n_val += 1
    avg_val = val_loss / n_val
    ae_val_losses.append(avg_val)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{AE_EPOCHS}  train_mse={avg_train:.6f}  val_mse={avg_val:.6f}")

print("Autoencoder training complete ✅")

In [ ]:
# === Autoencoder loss curves ===

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ae_train_losses, label='Train MSE', linewidth=2)
ax.plot(ae_val_losses, label='Val MSE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (Reconstruction Error)')
ax.set_title('Approach 2: Autoencoder Training (Real Images Only)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# === Visualize: how well does the AE reconstruct real vs fake? ===

autoencoder.eval()
n_show = 6

# Pick some real and fake test images
real_samples = real_test_all[:n_show]
fake_samples = fake_test[:n_show]

with torch.no_grad():
    real_recon = autoencoder(real_samples.to(device)).cpu()
    fake_recon = autoencoder(fake_samples.to(device)).cpu()

fig, axes = plt.subplots(4, n_show, figsize=(2.5 * n_show, 10))
row_labels = ['Real\nOriginal', 'Real\nReconstructed', 'Fake\nOriginal', 'Fake\nReconstructed']

for i in range(n_show):
    axes[0, i].imshow(real_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].imshow(real_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    real_err = F.mse_loss(real_recon[i], real_samples[i]).item()
    axes[1, i].set_title(f'MSE={real_err:.4f}', fontsize=8, color='green')

    axes[2, i].imshow(fake_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[3, i].imshow(fake_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    fake_err = F.mse_loss(fake_recon[i], fake_samples[i]).item()
    axes[3, i].set_title(f'MSE={fake_err:.4f}', fontsize=8, color='red')

for row in range(4):
    for i in range(n_show):
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(row_labels[row], fontsize=10, rotation=0,
                             labelpad=70, va='center')

plt.suptitle('Autoencoder Reconstruction: Real vs Fake', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Reconstruction-based anomaly scoring ===

autoencoder.eval()

@torch.no_grad()
def compute_recon_errors(model, images, device, batch_size=64):
    """Compute per-image MSE reconstruction error."""
    errors = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size].to(device)
        recon = model(batch)
        # Per-image MSE (mean over C, H, W)
        mse = ((recon - batch) ** 2).mean(dim=(1, 2, 3))
        errors.append(mse.cpu())
    return torch.cat(errors).numpy()

# Compute errors for real test images and fake test images
real_errors = compute_recon_errors(autoencoder, real_test_all, device)
fake_errors = compute_recon_errors(autoencoder, fake_test, device)

print(f"Real images — mean MSE: {real_errors.mean():.6f} ± {real_errors.std():.6f}")
print(f"Fake images — mean MSE: {fake_errors.mean():.6f} ± {fake_errors.std():.6f}")
print(f"Separation ratio: {fake_errors.mean() / real_errors.mean():.2f}x")

# Plot error distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(real_errors, bins=50, alpha=0.6, label=f'Real (n={len(real_errors)})', color='green', density=True)
ax.hist(fake_errors, bins=20, alpha=0.6, label=f'Fake (n={len(fake_errors)})', color='red', density=True)
ax.axvline(x=np.median(real_errors), color='green', linestyle='--', alpha=0.7, label=f'Real median: {np.median(real_errors):.5f}')
ax.axvline(x=np.median(fake_errors), color='red', linestyle='--', alpha=0.7, label=f'Fake median: {np.median(fake_errors):.5f}')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.set_title('Approach 2: Reconstruction Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# === Evaluate AE detector with same metrics as CNN ===

# Build labels and scores for the full test set
ae_labels = np.concatenate([
    np.zeros(len(real_errors)),  # real = 0
    np.ones(len(fake_errors)),   # fake = 1
])
ae_scores = np.concatenate([real_errors, fake_errors])

# Normalize scores to [0, 1] range for fair comparison
# Higher reconstruction error → more likely fake
ae_probs = (ae_scores - ae_scores.min()) / (ae_scores.max() - ae_scores.min())

ae_probs_out, ae_preds, ae_auc, ae_ap = evaluate_binary(
    ae_labels, ae_probs, "Approach 2: Reconstruction Anomaly Detector"
)

In [ ]:
# === AE detector gallery ===

# Build a pseudo test_ds for the gallery function
class SimpleDataset:
    def __init__(self, real_imgs, fake_imgs):
        self.images = torch.cat([real_imgs, fake_imgs], dim=0)

ae_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(ae_test_ds, ae_probs_out, ae_preds, ae_labels.astype(int),
             "Reconstruction Anomaly Detector")

---
## Head-to-Head Comparison

In [ ]:
# === Side-by-side comparison ===

print("=" * 60)
print("COMPARISON: CNN Discriminator vs Reconstruction Detector")
print("=" * 60)
print(f"{'Metric':<25} {'CNN Disc':>12} {'Recon AE':>12}")
print("-" * 60)
print(f"{'ROC-AUC':<25} {cnn_auc:>12.4f} {ae_auc:>12.4f}")
print(f"{'Avg Precision (PR-AUC)':<25} {cnn_ap:>12.4f} {ae_ap:>12.4f}")
print(f"{'Fake training samples':<25} {N_FAKE_TRAIN:>12d} {'0':>12s}")
print(f"{'Real training samples':<25} {len(real_train):>12d} {len(real_train):>12d}")
print()

# Overlay ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr_cnn, tpr_cnn, _ = roc_curve(cnn_labels, cnn_probs)
fpr_ae, tpr_ae, _ = roc_curve(ae_labels, ae_probs_out)
axes[0].plot(fpr_cnn, tpr_cnn, linewidth=2, label=f'CNN Disc (AUC={cnn_auc:.3f})')
axes[0].plot(fpr_ae, tpr_ae, linewidth=2, label=f'Recon AE (AUC={ae_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# PR
prec_cnn, rec_cnn, _ = precision_recall_curve(cnn_labels, cnn_probs)
prec_ae, rec_ae, _ = precision_recall_curve(ae_labels, ae_probs_out)
axes[1].plot(rec_cnn, prec_cnn, linewidth=2, label=f'CNN Disc (AP={cnn_ap:.3f})')
axes[1].plot(rec_ae, prec_ae, linewidth=2, label=f'Recon AE (AP={ae_ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Head-to-Head: CNN Discriminator vs Reconstruction Detector', fontsize=14)
plt.tight_layout()
plt.show()

## Discussion

### Approach 1: CNN Discriminator (Supervised)
- Trained with **60 fake samples** at a 20:1 imbalance — realistic for real-world fake detection
- Weighted BCE loss compensates for the rare fake class
- **Strength**: Directly optimized for the classification task
- **Weakness**: Needs labeled fakes, may overfit to this specific generator's artifacts

### Approach 2: Reconstruction Anomaly Detector (Representation Learning)
- Trained on **real images only** — no fakes needed at all
- The autoencoder learns the manifold of real fruit images
- Fakes are detected as off-manifold samples (high reconstruction error)
- **Strength**: Generalizes to *any* type of fake, not just this diffusion model's outputs
- **Weakness**: Relies on the assumption that fakes differ structurally from reals; may fail if fakes are very high quality

### Connection to Course Themes
- **Manifolds & VAEs**: The autoencoder's latent space *is* a learned manifold — real images cluster tightly, fakes are outliers
- **MAE / Representation Learning**: Reconstruction-based detection is the same principle as masked autoencoders — learn to reconstruct, detect anomalies by error
- **Adversarial Attacks**: A sufficiently good generator could fool the reconstruction detector by producing images that lie on the real manifold — this is the fundamental adversarial arms race